In [ ]:
import torch,os
from peft import LoraConfig, get_peft_model, TaskType
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
import pandas as pd

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
print(torch.cuda.is_available())

import re, os, string, random
import numpy as np

In [ ]:
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
import warnings
warnings.filterwarnings("ignore")

In [ ]:
sub_domain = "viabsa_rest" # rest_15, rest_16, rest_16_acos, laptop_16, viabsa_rest
domain = "dataset/" + sub_domain
#domain = "Dataset/vi_res"

domain_name = "restaurant"

path_train = domain + '/csv/train.csv'
df_train = pd.read_csv(path_train,index_col=False)
df_train.fillna("None", inplace=True)

path_dev = domain +  '/csv/dev.csv'
df_dev = pd.read_csv(path_dev,index_col=False)
df_dev.fillna("None", inplace=True)

path_test = domain + '/csv/test.csv'
df_test = pd.read_csv(path_test,index_col=False)
df_test.fillna("None", inplace=True)
df_test.head()

In [ ]:
prompt_type = "coding_format" # coding_format, paraphrase_format, free_order, json_format, original_format
model_id = "google/gemma-7b"
learning_rate = 2e-4
num_epochs = 5
batch_size = 4
rank = 8
seed = 42 # 42, 0, 1

"""Set the seed for reproducibility in PyTorch."""
random.seed(seed)            # Python random module.
np.random.seed(seed)         # Numpy module.
torch.manual_seed(seed)      # Sets the seed for generating random numbers.
torch.cuda.manual_seed(seed) # Sets the seed for CUDA.
torch.cuda.manual_seed_all(seed) # Sets the seed for all GPUs.
os.environ['PYTHONHASHSEED'] = str(seed)  # Set PYTHONHASHSEAT to prevent hash-based operations from randomness.

# Configure PyTorch to be deterministic
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
torch.backends.cudnn.deterministic = True  # Avoid nondeterministic algorithms.
torch.backends.cudnn.benchmark = False     # If the input sizes for your neural network do not vary, turning off benchmarking can improve reproducibility.
torch.use_deterministic_algorithms(True)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
cache_dir = "models/"

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir,
                                         local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", trust_remote_code=True,
                                                dtype=torch.bfloat16,
                                                attn_implementation='eager',
                                                #use_cache=False,
                                                force_download=False,
                                                cache_dir=cache_dir,
                                                local_files_only=True)

# Read training dataset

In [ ]:
from prompting import *

def format_prompt(input_review, prompt_type, domain_name):
    if prompt_type == "coding_format":
        prompt = CODING_FORMAT_PROMPTING

    elif prompt_type == "paraphrase_format":
        prompt = PARAPHRASE_FORMAT_PROMPTING
    
    elif prompt_type == "free_order":
        prompt = FREE_ORDER_FORMAT_PROMPTING
        
    elif prompt_type == "json_format":
        prompt = JSON_FORMAT_PROMPTING
        
    elif prompt_type == "original_format":
        prompt = ORIGINAL_FORMAT_PROMPTING
    else:
        print("ERROR")
    return prompt.format(input_review=input_review,domain_name=domain_name)

# Create format for input

In [ ]:
from datasets import Dataset, DatasetDict

def extract_input_output_for_training(df, type_dataset="train", prompt_type="code_format", domain_name="restaurant"):
    inputs = df["input"].tolist()
    print("Number of samples: ", len(inputs))
    if prompt_type == "coding_format":
        outputs = df["coding_format"].tolist()
    elif prompt_type == "paraphrase_format":
        outputs = df["paraphrase_format"].tolist()
    elif prompt_type == "free_order":
        outputs = df["free_order"].tolist()
    elif prompt_type == "json_format":
        outputs = df["json_format"].tolist()
    elif prompt_type == "original_format":
        outputs = df["output"].tolist()
    else:
        print("error")

    intput_list , output_list, model_input = [], [], []

    for index,review in enumerate(inputs):
        text_prompt = format_prompt(review, prompt_type, domain_name)
        intput_list.append(text_prompt)
        output_list.append(outputs[index])
        model_input.append(text_prompt + "" + outputs[index] + "<end_of_turn>")

    print(len(intput_list),len(output_list), len(model_input))

    print("======> Input: ", intput_list[0])
    print("======> Output: ", output_list[0])
    print("======> Model input: ", model_input[0])

    df_format = pd.DataFrame(list(zip(intput_list, output_list,model_input)), columns =['x_input', 'y_output', 'text'])
    dataset = DatasetDict()
    if type_dataset == "train":
        dataset['train'] = Dataset.from_pandas(df_format)
        return dataset, df_format
    else:
        return intput_list, output_list, model_input

In [ ]:
training_dataset,train_df_format = extract_input_output_for_training(df_train, type_dataset="train", prompt_type=prompt_type, domain_name=domain_name)
#======================================
from utils import calculate_max_input_length
max_input_length, max_output_length = calculate_max_input_length(tokenizer,
                                                                 train_df_format["text"].tolist(),
                                                                 train_df_format["y_output"].tolist())
print("max_input_length", max_input_length)
print("max_output_length", max_output_length)

In [ ]:
from peft import LoraConfig, PeftModel, TaskType
peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=rank, # Lora attention dimension.
        lora_alpha=rank*2, # the alpha parameter for Lora scaling.
        lora_dropout=0.1, # the dropout probability for Lora layers.
        bias="none",
        target_modules= "all-linear",
        #modules_to_save=["lm_head", "embed_token"],
        use_rslora=True,
)
model = get_peft_model(model, peft_config)

In [ ]:
# Print number of trainable and total parameters
def print_model_parameters_and_storage(model, bits=16):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    total_bytes = trainable_params * bits / 8
    total_mb = total_bytes / (1024 ** 2)
    
    def format_number(n):
        if n >= 1_000_000_000:
            return f"{n/1_000_000_000:.1f}B"
        elif n >= 1_000_000:
            return f"{n/1_000_000:.1f}M"
        elif n >= 1_000:
            return f"{n/1_000:.1f}K"
        return str(n)
    
    print(f"Total parameters: {format_number(total_params)}")
    print(f"Trainable parameters (LoRA): {format_number(trainable_params)}")
    print(f"Trainable ratio: {100 * trainable_params / total_params:.2f}%")
    print(f"Estimated LoRA storage: {total_mb:.2f} MB")
print_model_parameters_and_storage(model)

In [ ]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    processing_class=tokenizer,
    peft_config=peft_config,
    train_dataset=training_dataset["train"],
    args = SFTConfig(
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        warmup_steps=4,
        num_train_epochs = num_epochs, # Set this for 1 full training run.
        learning_rate = learning_rate,
        neftune_noise_alpha=5,
        bf16 = True,
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        load_best_model_at_end=False,
        seed = seed,
        output_dir = "checkpoint",
        report_to = "none",
        max_seq_length = max_input_length + max_output_length,
        dataset_num_proc = 4,
        logging_steps=50,
        packing = False, # Can make training 5x faster for short sequences.
    ),
)

In [ ]:
import time, os, warnings

start_time= time.time()
trainer.train()
stop_time=time.time()
time_training =stop_time - start_time
print("Training time (seconds): ", time_training)

# Evaluation

In [ ]:
x_test,y_test, mode_input_test = extract_input_output_for_training(df_test, type_dataset="test",prompt_type=prompt_type, domain_name=domain_name)

In [ ]:
from transformers import GenerationConfig
def evaluate_model_batch(samples,tokenizer,model, max_output_length):
    # Tokenize the batch of samples
    inputs = tokenizer(samples, return_tensors="pt", padding=True, truncation=True).to('cuda')

    # Generation configuration
    #generation_config  = GenerationConfig(
    #        do_sample=False,
    #        max_new_tokens=max_output_length,
    #        num_beams=5,
    #        eos_token_id=tokenizer.convert_tokens_to_ids("<end_of_turn>"),
    #        pad_token_id=tokenizer.convert_tokens_to_ids("<end_of_turn>"),
    #        return_full_text=False,
    #        early_stopping=True,
    #        stop_strings=['<end_of_turn>']
    #    )

    # Generation configuration
    generation_config = model.generation_config
    generation_config.do_sample = False
    generation_config.num_beams = 5
    generation_config.max_new_tokens = max_output_length

    # Generate predictions for the batch
    outputs = model.generate(**inputs, generation_config=generation_config,tokenizer=tokenizer)

    # Decode the outputs
    outputs_decode = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Post-process the generated outputs
    processed_outputs = []
    count = 0
    for item in outputs_decode:
        output = item.split("model\n")[1].strip()
        output = output.split("<end_of_turn>")[0].strip()
        if count == 0:
            #print("item: ", item)
            print("Output222: ", output)
            count+=1
            output = output.replace(")�", ")").replace(")*",")").strip()
        processed_outputs.append(output.strip())
    return processed_outputs

In [ ]:
from tqdm import tqdm
y_pred = []
start_time= time.time()
batch_size = 16
for i in tqdm(range(0, len(x_test), batch_size)):
    batch = x_test[i:i+batch_size]  # Create a batch of samples
    preds = evaluate_model_batch(batch, tokenizer, model, max_output_length)
    y_pred.extend(preds)
stop_time=time.time()
inference_time =stop_time - start_time
print("Inference time (seconds): ", inference_time)

In [ ]:
print(y_pred[0])
print(y_test[0])

In [ ]:
from utils import *

list_y_test = convert_predict_to_json_format(y_test,prompt_type)
list_y_pred = convert_predict_to_json_format(y_pred,prompt_type)

print(len(list_y_test), len(list_y_pred))

In [ ]:
index = 5
list_y_test[index]

In [ ]:
list_y_pred[index]

In [ ]:
from eval_utils import *
scores, all_labels, all_preds = compute_scores(list_y_pred,list_y_test)
print("Results:")
print(scores)

In [ ]:
df_test["prediction"] = all_preds
df_test["groundtruth"] = all_labels
text_print = "Model:\t" + model_id + "\t" + prompt_type + " - seed: " + str(seed) + "\n" + "Performance:\t" + str(scores) + "\n\n"

try:
    if str(domain) != "dataset/viabsa_rest":
        df_test.to_csv("prediction/en/"+ model_id.split("/")[-1] + "_" + sub_domain + str(prompt_type) + "_prediction.csv",index=False)
        with open("scores/" + model_id.split("/")[-1] + "_" +sub_domain + ".txt", 'a', encoding='utf8') as file:
            file.write(text_print)
    else:
        print("True")
        df_test.to_csv("prediction/en/"+ model_id.split("/")[-1] + "_" + str(prompt_type) + "_prediction.csv",index=False)
        with open("scores/" + model_id.split("/")[-1] + "_vi_res" + ".txt", 'a', encoding='utf8') as file:
            file.write(text_print)
except:
    if str(domain) == "dataset/viabsa_rest":
        print("True")
        with open("scores/" + model_id.split("/")[-1] + "_vi_res" + ".txt", 'a', encoding='utf8') as file:
            file.write(text_print)
    else:
        with open("scores/" + model_id.split("/")[-1]  + ".txt", 'a', encoding='utf8') as file:
            file.write(text_print)

In [ ]:
all_preds[0]

In [ ]:
df_b = pd.DataFrame({
    "text": df_test["input"].tolist(),
    "prediction": all_preds,
    "groundtruth": all_labels,
})
df_b.to_csv("prediction/ViAbSQA"+prompt_type+".csv", index=False)
df_b.head()